# Data Preprocessing Pipeline

This notebook handles comprehensive data preprocessing including:
- Data loading and validation
- Missing value handling
- Feature scaling and normalization
- Train-test split
- Feature engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

## 1. Data Loading

In [ ]:
df = pd.read_csv('../Data/Spotify_Youtube.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
print('Data Info:')
df.info()

## 2. Data Cleaning

In [ ]:
df.drop(columns=['Unnamed: 0'], inplace=True)
print('✓ Dropped index column')

In [ ]:
print('Missing values before cleaning:')
print(df.isnull().sum())

In [ ]:
audio_features = ['Danceability', 'Energy', 'Key', 'Loudness', 'Speechiness',
                  'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo', 'Duration_ms']

df.dropna(subset=audio_features, inplace=True)
print(f'✓ Dropped rows with missing audio features. Shape: {df.shape}')

In [ ]:
imputer_numeric = SimpleImputer(strategy='median')
numeric_cols = ['Views', 'Likes', 'Comments', 'Stream']
df[numeric_cols] = imputer_numeric.fit_transform(df[numeric_cols])
print('✓ Imputed missing numeric values with median')

In [ ]:
imputer_categorical = SimpleImputer(strategy='most_frequent')
categorical_cols = ['Url_youtube', 'Title', 'Channel', 'Description', 'Licensed', 'official_video']
df[categorical_cols] = imputer_categorical.fit_transform(df[categorical_cols])
print('✓ Imputed missing categorical values')

In [ ]:
df.drop_duplicates(subset=['Track', 'Artist'], inplace=True, keep='first')
print(f'✓ Removed duplicates. Shape: {df.shape}')

## 3. Data Type Conversion

In [ ]:
df['official_video'] = df['official_video'].astype('bool')
df['Licensed'] = df['Licensed'].astype('bool')
print('✓ Converted boolean columns')

In [ ]:
df['Description'] = df['Description'].str.replace('\n', ' ', regex=False)
print('✓ Cleaned Description column')

## 4. Feature Engineering

In [ ]:
df['Engagement_Rate'] = (df['Likes'] / (df['Views'] + 1)) * 100
df['Discussion_Rate'] = (df['Comments'] / (df['Views'] + 1)) * 100
df['Like_Comment_Ratio'] = (df['Likes'] / (df['Comments'] + 1))
print('✓ Created engagement features')

In [ ]:
df['Danceability'] = df['Danceability'] * 100
df['Energy'] = df['Energy'] * 100
df['Views'] = df['Views'] / 1e6
df['Stream'] = df['Stream'] / 1e6
df['Likes'] = df['Likes'] / 1e6
df['Comments'] = df['Comments'] / 1e3
print('✓ Normalized scale columns')

## 5. Select Features for ML

In [ ]:
feature_cols = ['Danceability', 'Energy', 'Key', 'Loudness', 'Speechiness',
                'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo',
                'Duration_ms', 'Views', 'Likes', 'Comments', 'Engagement_Rate',
                'Discussion_Rate', 'Like_Comment_Ratio']

target_col = 'Stream'

X = df[feature_cols].copy()
y = df[target_col].copy()

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeatures: {feature_cols}')

## 6. Handle Outliers (Optional)

In [ ]:
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ~((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)
print(f'Rows before outlier removal: {len(X)}')
X = X[outlier_mask]
y = y[outlier_mask]
print(f'Rows after outlier removal: {len(X)}')

## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
print('✓ Features scaled using StandardScaler')
print(f'\nScaled features sample:')
X_scaled.head()

## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nFeatures per set: {X_train.shape[1]}')

## 9. Save Processed Data

In [ ]:
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

pd.DataFrame({'feature_cols': feature_cols}).to_csv('data/feature_cols.csv', index=False)

print('✓ Saved processed data:')
print('  - X_train.csv')
print('  - X_test.csv')
print('  - y_train.csv')
print('  - y_test.csv')
print('  - feature_cols.csv')

In [ ]:
import pickle
with open('data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✓ Saved scaler for future predictions')

## 10. Data Summary

In [ ]:
print('=== PREPROCESSING SUMMARY ===')
print(f'\nOriginal dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Final dataset: {len(X)} rows × {X.shape[1]} features')
print(f'\nTarget variable (Streams) statistics:')
print(y.describe())
print(f'\nFeatures statistics:')
X_scaled.describe()